In [ ]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from reprojection import *
from show import *

In [ ]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos_/*')))
#files = np.array(sorted(glob.glob('C:/Users/ikiru/data/4science/blos/*')))
#files = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/2315/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])

#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,8,31)], axis=0))[0] # CR2314

t = np.where(np.all([dates > datetime(2026,8,31),
                     dates < datetime(2026,9,29)], axis=0))[0] # CR2315

#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,9,29)], axis=0))[0] # CR2314 + CR2315

files = files[t]
files[0], files[-1], len(files)

In [ ]:
data, variance, view = make_map(files, correct_dr=True, kind='south', mu_thr=0.1)

In [ ]:
fig, ax = show_data(data, view, cmap='seismic', vmin=-20, vmax=20)

In [ ]:
WSID = 360 / 25.38
A, B, C = 14.712, -2.396, -1.787
RSUN = 696e8

def diff_rot(r, *, a=A-WSID, b=B, c=C, u0=1000, lat0=45, q=0.5):
    def f(x):
        return np.sin(2 * x * np.pi / 180) + q * np.sin(4 * x * np.pi / 180) + (2 * q - 1) / 3 * np.sin(6 * x * np.pi / 180)

    theta, phi = r
    y = np.sin(theta * np.pi / 180)

    u = u0 / RSUN * f(theta) / f(lat0) * 180 / np.pi * 24 * 3600
    return np.array([u, (a + b * y ** 2 + c * y ** 4)])

transform = ~(view.transform() + ToSpherical())

a = np.random.uniform([-90,0],[90,360], size=(1000,2)).T
w = diff_rot(a)
b = a + w

a = np.array(transform(*a))
w = np.array(transform(*b)) - a

In [ ]:
def step(func, r, h):
    k1 = func(r)
    k2 = func(r + k1 * h / 2)
    k3 = func(r + k2 * h / 2)
    k4 = func(r + k3 * h)
    return r + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

def rk4(func, r, h, n):
    out = [r]
    for i in range(n):
        out.append(step(func, out[-1], h))
    return np.array(out)


In [ ]:
n = 500

#phi = np.random.uniform(0, 360, size=n)
phi = np.linspace(0, 360, n)
theta = np.random.normal(0, 30, size=n)
#theta = np.random.uniform(-90, 90, size=n)
a = np.array((theta, phi))


q = np.transpose(rk4(diff_rot, a, 1e-1, 300), (1,0,2))
q = np.array(transform(*q))

In [ ]:
fig, ax = show_data(data, view, cmap='seismic', vmin=-20, vmax=20)
#fig, ax = show_data(data, view, cmap='hmimag', vmin=-250, vmax=250)
ax.plot(q[1], q[0], color='black', lw=0.7, alpha=0.7)
#ax.quiver(a[1], a[0], w[1], w[0], width=1e-3, scale=0.1, scale_units='xy', alpha=0.7)

In [ ]:
from scipy.integrate import RK23